In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

from src.utils import (
    load_binary_metadata,
    compute_metrics,
    print_metrics,
)

In [3]:
train_df, val_df = load_binary_metadata(csv_path="../binary_vqa_metadata.csv")

print("Train size: ", len(train_df))
print("Validation size: ", len(val_df))
print("\nColumns: ", train_df.columns.tolist())
print("\nSample question: ")
print(train_df["question"].head(3).tolist())

Train size:  750
Validation size:  190

Columns:  ['question', 'answer', 'label', 'question_type', 'split', 'inferred_modality']

Sample question: 
['does this represent infectious process?', 'is the vertebro-basilar arterial network viewed in this section?', 'is this an ap image?']


In [14]:
#Extract text and labels
X_train = train_df["question"]
y_train = train_df["label"]
X_val = val_df["question"]
y_val = val_df["label"]

#Build a pipeline: TF-IDF -> Logistic Regression
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

#Train the model
lr_pipeline.fit(X_train, y_train)
print("Training completed.")

Training completed.


In [16]:
# Predict on validation set
y_pred = lr_pipeline.predict(X_val)
y_prob = lr_pipeline.predict_proba(X_val)[:, 1]

# Compute and print metrics
metrics_lr = compute_metrics(y_val, y_pred, y_prob)
print_metrics(metrics_lr, model_name="TF-IDF + Logistic Regression")


  TF-IDF + Logistic Regression
  Accuracy : 0.6579
  F1 Score : 0.6369
  AUC-ROC  : 0.7174



In [17]:
#Build a pipeline: TF-IDF -> SVM
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ("clf", SVC(kernel="linear", probability=True, random_state=42)),
])

#Train the model
svm_pipeline.fit(X_train, y_train)
print("Training completed.")

Training completed.


In [18]:
# Predict
y_pred_svm = svm_pipeline.predict(X_val)
y_prob_svm = svm_pipeline.predict_proba(X_val)[:, 1]

# Compute and print metrics
metrics_svm = compute_metrics(y_val, y_pred_svm, y_prob_svm)
print_metrics(metrics_svm, model_name="TF-IDF + SVM")


  TF-IDF + SVM
  Accuracy : 0.6842
  F1 Score : 0.6667
  AUC-ROC  : 0.7557



In [ ]:
summary = pd.DataFrame([
    {"Model": "TF-IDF + Logistic Regression", **metrics_lr},
    {"Model": "TF-IDF + SVM",                **metrics_svm},
])

print(summary.to_string(index=False))